# Combinatorial CoWork 2026 — Notebook 03: Change of posets and common refinement

Audience:
- Participants who want to see functoriality, transport, and common refinement as first-class workflow tasks rather than ad hoc storage manipulations.

Learning goals:
- Use `common_refinement(...)` as the canonical notebook-facing transport path.
- Inspect the translated modules and the projection maps without unpacking fields manually.
- Demonstrate `restriction`, `pushforward_left`, `pushforward_right`, and their derived versions on a tiny explicit finite map.


## Outline

1. Setup and two separately encoded box examples
2. Translate them to a common refinement
3. Inspect the projection maps and restrictions
4. Use `hom(...; transport=:common_refinement)`
5. One clearly labeled advanced aside for explicit pushforwards on a tiny finite map


In [17]:
NOTEBOOK_STEM = "03_change_of_posets_and_common_refinement"

_TO_ROOT = let
    dir = abspath(pwd())
    root = nothing
    while true
        if isfile(joinpath(dir, "Project.toml")) && isfile(joinpath(dir, "src", "TamerOp.jl"))
            root = dir
            break
        end
        parent = dirname(dir)
        parent == dir && error("Could not locate repo root containing Project.toml and src/TamerOp.jl from pwd()=$(pwd()).")
        dir = parent
    end
    root
end

import Pkg
Pkg.activate(_TO_ROOT; io=devnull)

if !isdefined(Main, :TamerOp)
    @eval Main using TamerOp
end

TO = Main.TamerOp
CM = TO.CoreModules
OPT = TO.Options
sc = TO.SessionCache()
outdir = joinpath(_TO_ROOT, "examples", "_outputs", "combinatorial_cowork_2026", NOTEBOOK_STEM)
mkpath(outdir)


"/home/eriknovak/Documents/duke_fall_2025/tamer-op/examples/_outputs/combinatorial_cowork_2026/03_change_of_posets_and_common_refinement"

## 1. Two small box presentations that do not share an encoding a priori

We encode them separately on purpose so that common refinement has real work to do.

The point of this notebook is not “here are two modules already on the same poset.” The point is “here are two independently encoded objects, and the transport story itself should still be notebook-facing and explicit.”


In [18]:
SD = TO.SyntheticData
box_left = SD.box_bar_fringe(
    bars=[([0.0, 0.0], [1.25, 1.0]), ([1.0, 0.25], [2.0, 1.5])],
    field=CM.QQField(),
)
box_right = SD.staircase_box_fringe(
    bars=[([0.0, 0.0], [1.5, 1.0]), ([0.75, 0.25], [2.25, 1.75]), ([1.5, 0.75], [3.0, 2.5])],
    field=CM.QQField(),
)


SyntheticBoxFringe
  ambient_dim: 2
  nupsets: 3
  ndownsets: 3
  matrix_size: (3, 3)
  field: TamerOp.CoreModules.CoeffFields.QQField()

Start with the source objects themselves.

The two presentations use different synthetic families, so even before encoding they carry slightly different geometry. The relevant notebook question is still the generic one: what did we build?


In [19]:
(; left_source=TO.describe(box_left),
   right_source=TO.describe(box_right))


(left_source = (kind = :synthetic_box_fringe, ambient_dim = 2, nupsets = 2, ndownsets = 2, matrix_size = (2, 2), field = TamerOp.CoreModules.CoeffFields.QQField()), right_source = (kind = :synthetic_box_fringe, ambient_dim = 2, nupsets = 3, ndownsets = 3, matrix_size = (3, 3), field = TamerOp.CoreModules.CoeffFields.QQField()))

Now encode them separately. Because the encodings are independent, we should not expect them to share a classifier map or a finite poset automatically.


In [20]:
enc_left = TO.encode(box_left; cache=sc)
enc_right = TO.encode(box_right; cache=sc)
left_regions_spec = TO.Visualization.visual_spec(TO.encoding_map(enc_left); kind=:regions)


VisualizationSpec
  visual_kind: regions
  title: "Box encoding"
  subtitle: ""
  nlayers: 19
  npanels: 0
  layer_types: (:RectLayer, :SegmentLayer, :RectLayer, :SegmentLayer, :RectLayer, :SegmentLayer, :RectLayer, :SegmentLayer, :RectLayer, :SegmentLayer, :RectLayer, :SegmentLayer, :RectLayer, :SegmentLayer, :RectLayer, :SegmentLayer, :RectLayer, :SegmentLayer, :PointLayer)
  axes: (xlabel = "x1", ylabel = "x2", xlimits = (-2.0, 4.0), ylimits = (-2.0, 1.625), zlabel = "z", zlimits = nothing, aspect = :data, xticks = nothing, yticks = nothing)
  metadata: (object = :pl_boxes, nregions = 9, ncells = 25, query_count = 0, figure_size = (860, 620), legend_position = :right)

Inspect the two encoded objects before transporting them.

The key point is that these are already perfectly valid `EncodingResult`s, but they are not aligned yet. That is exactly the situation where `common_refinement(...)` should be the first remembered workflow tool.


In [21]:
(; left=TO.describe(enc_left),
   right=TO.describe(enc_right),
   same_poset=(TO.encoding_poset(enc_left) === TO.encoding_poset(enc_right)),
   same_classifier=(TO.encoding_map(enc_left) === TO.encoding_map(enc_right)),
   left_regions_summary=TO.Visualization.visual_summary(left_regions_spec))


(left = (kind = :encoding_result, poset_type = TamerOp.ZnEncoding.SignaturePoset{1, 1}, module_type = TamerOp.Modules.PModule{Rational{BigInt}, TamerOp.CoreModules.CoeffFields.QQField, Matrix{Rational{BigInt}}, TamerOp.ZnEncoding.SignaturePoset{1, 1}}, encoding_map_type = TamerOp.EncodingCore.CompiledEncoding{TamerOp.PLBackend.PLEncodingMapBoxes{2, 1, 1}, TamerOp.ZnEncoding.SignaturePoset{1, 1}, Tuple{Vector{Float64}, Vector{Float64}}, Vector{Tuple{Float64, Float64}}, TamerOp.CoreModules.EncodingCache}, compiled = true, backend = :pl_backend, has_cohomology = true, has_presentation = true, module_dims = [0, 0, 0, 1, 0, 0, 2, 1, 0]), right = (kind = :encoding_result, poset_type = TamerOp.ZnEncoding.SignaturePoset{1, 1}, module_type = TamerOp.Modules.PModule{Rational{BigInt}, TamerOp.CoreModules.CoeffFields.QQField, Matrix{Rational{BigInt}}, TamerOp.ZnEncoding.SignaturePoset{1, 1}}, encoding_map_type = TamerOp.EncodingCore.CompiledEncoding{TamerOp.PLBackend.PLEncodingMapBoxes{2, 1, 1}, T

## 2. Common refinement is the canonical notebook-facing transport path

This is the surface to remember when two encoded modules do not already share a classifier map.

The result of `common_refinement(...)` is a typed transport object. It keeps the common poset, the two projection maps, and the translated modules together so the notebook does not have to reconstruct that story by hand.


In [22]:
cref = TO.common_refinement(enc_left, enc_right; cache=sc)


CommonRefinementTranslationResult
  field: TamerOp.CoreModules.CoeffFields.QQField()
  common_nvertices: 135
  left_nvertices: 9
  right_nvertices: 15
  left_total_dim: 60
  right_total_dim: 63
  translated_on_common_poset: true

First inspect the common-refinement result itself.

The important questions are:

- what common poset did we build?
- what translated objects does this give us?
- what projection maps does it remember for later functoriality operations?


In [23]:
mods = TO.translated_modules(cref)
pis = TO.projection_maps(cref)

(; common_refinement=TO.describe(cref),
   common_poset_summary=TO.describe(TO.common_poset(cref)),
   left_translated=TO.describe(mods.left),
   right_translated=TO.describe(mods.right))


(common_refinement = (kind = :common_refinement_translation_result, field = TamerOp.CoreModules.CoeffFields.QQField(), common_nvertices = 135, left_nvertices = 9, right_nvertices = 15, left_total_dim = 60, right_total_dim = 63, translated_on_common_poset = true), common_poset_summary = (kind = :product_poset, nvertices = 135, left_nvertices = 9, right_nvertices = 15), left_translated = (kind = :pmodule, field = "QQ", vertices = 135, total_dimension = 60, maximum_stalk = 2, edge_count = 378, poset_type = :ProductPoset), right_translated = (kind = :pmodule, field = "QQ", vertices = 135, total_dimension = 63, maximum_stalk = 2, edge_count = 378, poset_type = :ProductPoset))

## 3. Inspect the projection maps and restrictions

The projection maps returned by `common_refinement(...)` are part of the mathematical payload, not hidden implementation detail.

Here it is useful to inspect their direction explicitly:

- `source_poset(pi)` is where restriction lands,
- `target_poset(pi)` is the original ambient poset the map points into.

That is why restricting along these maps produces modules on the common poset.


In [24]:
(; left_projection=TO.describe(pis.left),
   right_projection=TO.describe(pis.right),
   left_source_matches_common=(TO.Encoding.source_poset(pis.left) === TO.common_poset(cref)),
   right_source_matches_common=(TO.Encoding.source_poset(pis.right) === TO.common_poset(cref)),
   left_target_matches_original=(TO.Encoding.target_poset(pis.left) === TO.encoding_poset(enc_left)),
   right_target_matches_original=(TO.Encoding.target_poset(pis.right) === TO.encoding_poset(enc_right)))


(left_projection = (kind = :finite_encoding_map, source_poset_kind = :ProductPoset, target_poset_kind = :SignaturePoset, nsource = 135, ntarget = 9, image_size = 9, inverse_fibers_cached = false, label_cache_cached = false), right_projection = (kind = :finite_encoding_map, source_poset_kind = :ProductPoset, target_poset_kind = :SignaturePoset, nsource = 135, ntarget = 15, image_size = 15, inverse_fibers_cached = false, label_cache_cached = false), left_source_matches_common = true, right_source_matches_common = true, left_target_matches_original = true, right_target_matches_original = true)

Restriction is the pullback-side operation. In this notebook it should reproduce the translated modules that `common_refinement(...)` already packaged for us.


In [25]:
left_pull = TO.restriction(pis.left, enc_left; cache=sc)
right_pull = TO.restriction(pis.right, enc_right; cache=sc)


ModuleTranslationResult
  translation_kind: restriction
  module_type: TamerOp.Modules.PModule{Rational{BigInt}, TamerOp.CoreModules.CoeffFields.QQField, Matrix{Rational{BigInt}}, TamerOp.FiniteFringe.ProductPoset{TamerOp.ZnEncoding.SignaturePoset{1, 1}, TamerOp.ZnEncoding.SignaturePoset{1, 1}}}
  map_type: TamerOp.Encoding.EncodingMap{TamerOp.FiniteFringe.ProductPoset{TamerOp.ZnEncoding.SignaturePoset{1, 1}, TamerOp.ZnEncoding.SignaturePoset{1, 1}}, TamerOp.ZnEncoding.SignaturePoset{1, 1}}
  has_classifier: false
  source_type: EncodingResult{TamerOp.ZnEncoding.SignaturePoset{1, 1}, TamerOp.Modules.PModule{Rational{BigInt}, TamerOp.CoreModules.CoeffFields.QQField, Matrix{Rational{BigInt}}, TamerOp.ZnEncoding.SignaturePoset{1, 1}}, TamerOp.EncodingCore.CompiledEncoding{TamerOp.PLBackend.PLEncodingMapBoxes{2, 1, 1}, TamerOp.ZnEncoding.SignaturePoset{1, 1}, Tuple{Vector{Float64}, Vector{Float64}}, Vector{Tuple{Float64, Float64}}, TamerOp.CoreModules.EncodingCache}, TamerOp.FiniteFringe.F

The check here is not deep categorical equality of every internal object. The notebook-level question is whether the pullback results live on the common poset and look like the translated modules we expected.


In [26]:
(; left_pull=TO.describe(left_pull),
   right_pull=TO.describe(right_pull),
   translated_left_matches_poset=(TO.encoding_poset(left_pull) === TO.common_poset(cref)),
   translated_right_matches_poset=(TO.encoding_poset(right_pull) === TO.common_poset(cref)),
   common_hom_dim=TO.hom_dimension(enc_left, enc_right; transport=:common_refinement, cache=sc),
   common_hom=TO.describe(TO.hom(enc_left, enc_right; transport=:common_refinement, cache=sc)))


(left_pull = (kind = :module_translation_result, translation_kind = :restriction, module_type = TamerOp.Modules.PModule{Rational{BigInt}, TamerOp.CoreModules.CoeffFields.QQField, Matrix{Rational{BigInt}}, TamerOp.FiniteFringe.ProductPoset{TamerOp.ZnEncoding.SignaturePoset{1, 1}, TamerOp.ZnEncoding.SignaturePoset{1, 1}}}, map_type = TamerOp.Encoding.EncodingMap{TamerOp.FiniteFringe.ProductPoset{TamerOp.ZnEncoding.SignaturePoset{1, 1}, TamerOp.ZnEncoding.SignaturePoset{1, 1}}, TamerOp.ZnEncoding.SignaturePoset{1, 1}}, has_classifier = false, source_type = EncodingResult{TamerOp.ZnEncoding.SignaturePoset{1, 1}, TamerOp.Modules.PModule{Rational{BigInt}, TamerOp.CoreModules.CoeffFields.QQField, Matrix{Rational{BigInt}}, TamerOp.ZnEncoding.SignaturePoset{1, 1}}, TamerOp.EncodingCore.CompiledEncoding{TamerOp.PLBackend.PLEncodingMapBoxes{2, 1, 1}, TamerOp.ZnEncoding.SignaturePoset{1, 1}, Tuple{Vector{Float64}, Vector{Float64}}, Vector{Tuple{Float64, Float64}}, TamerOp.CoreModules.EncodingCache

## 4. Advanced aside: one tiny explicit finite map for pushforwards

This is the only deliberate advanced cell in the core suite.

We build a two-point chain onto a one-point chain so that the left/right pushforward story is completely explicit. This is not the canonical everyday notebook path, but it is the cleanest way to make the functorial operations concrete for this audience.


In [27]:
TOA = TO.Advanced
Psmall = TOA.FinitePoset(reshape(Bool[1], 1, 1); check=false)
Qbig = TOA.FinitePoset(Bool[1 1; 0 1]; check=false)

pi_q_to_p = TOA.EncodingMap(Qbig, Psmall, [1, 1])
id_P = TOA.EncodingMap(Psmall, Psmall, [1])
id_Q = TOA.EncodingMap(Qbig, Qbig, [1, 2])


EncodingMap
  source: TamerOp.FiniteFringe.FinitePoset with 2 vertices
  target: TamerOp.FiniteFringe.FinitePoset with 2 vertices
  image_size: 2
  inverse_fibers_cached: false
  label_cache_cached: false

Before pushing anything forward, inspect the explicit map and the two tiny encoded modules.

This cell is doing the same thing as the earlier notebook sections: make the object contract visible before asking for a heavier construction.


In [28]:
field = CM.QQField()
H_small = TOA.one_by_one_fringe(Psmall, TOA.principal_upset(Psmall, 1), TOA.principal_downset(Psmall, 1), TO.QQ(1); field=field)
H_big = TOA.one_by_one_fringe(Qbig, TOA.principal_upset(Qbig, 1), TOA.principal_downset(Qbig, 2), TO.QQ(1); field=field)

enc_small = TO.EncodingResult(Psmall, TOA.pmodule_from_fringe(H_small), id_P; H=H_small, opts=OPT.EncodingOptions(field=field), backend=:workshop)
enc_big = TO.EncodingResult(Qbig, TOA.pmodule_from_fringe(H_big), id_Q; H=H_big, opts=OPT.EncodingOptions(field=field), backend=:workshop)

(; enc_small=TO.describe(enc_small),
   enc_big=TO.describe(enc_big),
   pi_map=TO.describe(pi_q_to_p),
   map_source=TO.describe(TO.Encoding.source_poset(pi_q_to_p)),
   map_target=TO.describe(TO.Encoding.target_poset(pi_q_to_p)))


(enc_small = (kind = :encoding_result, poset_type = TamerOp.FiniteFringe.FinitePoset, module_type = TamerOp.Modules.PModule{Rational{BigInt}, TamerOp.CoreModules.CoeffFields.QQField, Matrix{Rational{BigInt}}, TamerOp.FiniteFringe.FinitePoset}, encoding_map_type = TamerOp.Encoding.EncodingMap{TamerOp.FiniteFringe.FinitePoset, TamerOp.FiniteFringe.FinitePoset}, compiled = false, backend = :workshop, has_cohomology = true, has_presentation = false, module_dims = [1]), enc_big = (kind = :encoding_result, poset_type = TamerOp.FiniteFringe.FinitePoset, module_type = TamerOp.Modules.PModule{Rational{BigInt}, TamerOp.CoreModules.CoeffFields.QQField, Matrix{Rational{BigInt}}, TamerOp.FiniteFringe.FinitePoset}, encoding_map_type = TamerOp.Encoding.EncodingMap{TamerOp.FiniteFringe.FinitePoset, TamerOp.FiniteFringe.FinitePoset}, compiled = false, backend = :workshop, has_cohomology = true, has_presentation = false, module_dims = [1, 1]), pi_map = (kind = :finite_encoding_map, source_poset_kind = :

## 5. Left and right pushforwards

Now that the map is explicit, the two direct pushforwards read like ordinary functorial constructions.

- `pushforward_left(...)` and `pushforward_right(...)` need not agree.
- The point of this tiny example is to let the audience see those two outputs side by side rather than as abstract notation.


In [29]:
left = TO.pushforward_left(pi_q_to_p, enc_big; cache=sc)
right = TO.pushforward_right(pi_q_to_p, enc_big; cache=sc)

(; left=TO.describe(left),
   right=TO.describe(right))


(left = (kind = :module_translation_result, translation_kind = :pushforward_left, module_type = TamerOp.Modules.PModule{Rational{BigInt}, TamerOp.CoreModules.CoeffFields.QQField, SparseArrays.SparseMatrixCSC{Rational{BigInt}, Int64}, TamerOp.FiniteFringe.FinitePoset}, map_type = TamerOp.Encoding.EncodingMap{TamerOp.FiniteFringe.FinitePoset, TamerOp.FiniteFringe.FinitePoset}, has_classifier = false, source_type = EncodingResult{TamerOp.FiniteFringe.FinitePoset, TamerOp.Modules.PModule{Rational{BigInt}, TamerOp.CoreModules.CoeffFields.QQField, Matrix{Rational{BigInt}}, TamerOp.FiniteFringe.FinitePoset}, TamerOp.Encoding.EncodingMap{TamerOp.FiniteFringe.FinitePoset, TamerOp.FiniteFringe.FinitePoset}, TamerOp.FiniteFringe.FringeModule{Rational{BigInt}, TamerOp.FiniteFringe.FinitePoset, TamerOp.CoreModules.CoeffFields.QQField, SparseArrays.SparseMatrixCSC{Rational{BigInt}, Int64}}, Nothing, @NamedTuple{}}, module_dims = [1]), right = (kind = :module_translation_result, translation_kind = :p

## 6. Derived pushforwards

The derived pushforwards are the homological correction terms for those direct functors.

We keep `maxdeg=1` so the notebook stays small. The result is a vector of translated-result objects indexed by derived degree.


In [30]:
df_opts = OPT.DerivedFunctorOptions(maxdeg=1)
dleft = TO.derived_pushforward_left(pi_q_to_p, enc_big; opts=df_opts, cache=sc)
dright = TO.derived_pushforward_right(pi_q_to_p, enc_big; opts=df_opts, cache=sc)

(; derived_left=map(TO.describe, dleft),
   derived_right=map(TO.describe, dright))


(derived_left = @NamedTuple{kind::Symbol, translation_kind::Symbol, module_type::DataType, map_type::DataType, has_classifier::Bool, source_type::DataType, module_dims::Vector{Int64}}[(kind = :module_translation_result, translation_kind = :derived_pushforward_left, module_type = TamerOp.Modules.PModule{Rational{BigInt}, TamerOp.CoreModules.CoeffFields.QQField, Matrix{Rational{BigInt}}, TamerOp.FiniteFringe.FinitePoset}, map_type = TamerOp.Encoding.EncodingMap{TamerOp.FiniteFringe.FinitePoset, TamerOp.FiniteFringe.FinitePoset}, has_classifier = 0, source_type = EncodingResult{TamerOp.FiniteFringe.FinitePoset, TamerOp.Modules.PModule{Rational{BigInt}, TamerOp.CoreModules.CoeffFields.QQField, Matrix{Rational{BigInt}}, TamerOp.FiniteFringe.FinitePoset}, TamerOp.Encoding.EncodingMap{TamerOp.FiniteFringe.FinitePoset, TamerOp.FiniteFringe.FinitePoset}, TamerOp.FiniteFringe.FringeModule{Rational{BigInt}, TamerOp.FiniteFringe.FinitePoset, TamerOp.CoreModules.CoeffFields.QQField, SparseArrays.Sp

## 7. Export the region views

Use the same `save_visuals(...)` surface here so participants see that export is uniform across workflow objects.

The exported figures are intentionally small: one regions view for each independently encoded source.


In [31]:
exports = TO.save_visuals(
    outdir,
    [
        (; stem="left_regions", obj=TO.encoding_map(enc_left), kind=:regions),
        (; stem="right_regions", obj=TO.encoding_map(enc_right), kind=:regions),
    ];
    format=:html,
    backend=:cairomakie,
)

Dict(TO.export_stem(r) => TO.export_path(r) for r in exports)


Dict{String, String} with 2 entries:
  "right_regions" => "/home/eriknovak/Documents/duke_fall_2025/tamer-op/example…
  "left_regions"  => "/home/eriknovak/Documents/duke_fall_2025/tamer-op/example…

## Try this next

Change one bar in `box_right`, rerun the common-refinement cells, and compare:

- the translated-module summaries,
- the projection-map summaries,
- and `common_hom_dim`.

The point is to see which parts of the workflow stay stable and which parts genuinely depend on the transport geometry.


In [32]:
box_right_variant = SD.staircase_box_fringe(
    bars=[([0.0, 0.0], [1.25, 1.0]), ([0.75, 0.25], [2.5, 1.5]), ([1.75, 1.0], [3.25, 2.75])],
    field=CM.QQField(),
)
enc_right_variant = TO.encode(box_right_variant; cache=sc)
cref_variant = TO.common_refinement(enc_left, enc_right_variant; cache=sc)

(; right_variant=TO.describe(enc_right_variant),
   variant_common_refinement=TO.describe(cref_variant),
   variant_common_hom_dim=TO.hom_dimension(enc_left, enc_right_variant; transport=:common_refinement, cache=sc))


(right_variant = (kind = :encoding_result, poset_type = TamerOp.ZnEncoding.SignaturePoset{1, 1}, module_type = TamerOp.Modules.PModule{Rational{BigInt}, TamerOp.CoreModules.CoeffFields.QQField, Matrix{Rational{BigInt}}, TamerOp.ZnEncoding.SignaturePoset{1, 1}}, encoding_map_type = TamerOp.EncodingCore.CompiledEncoding{TamerOp.PLBackend.PLEncodingMapBoxes{2, 1, 1}, TamerOp.ZnEncoding.SignaturePoset{1, 1}, Tuple{Vector{Float64}, Vector{Float64}}, Vector{Tuple{Float64, Float64}}, TamerOp.CoreModules.EncodingCache}, compiled = true, backend = :pl_backend, has_cohomology = true, has_presentation = true, module_dims = [0, 0, 0, 0, 1, 0, 0, 0, 2, 1, 0, 0, 2, 1, 0]), variant_common_refinement = (kind = :common_refinement_translation_result, field = TamerOp.CoreModules.CoeffFields.QQField(), common_nvertices = 135, left_nvertices = 9, right_nvertices = 15, left_total_dim = 60, right_total_dim = 63, translated_on_common_poset = true), variant_common_hom_dim = 0)